In first example, we will design a neural network that can discover binding motifs in DNA based on the results of an assay that determines whether a longer DNA sequence binds to the protein or not. Here, the longer DNA sequences are our independent variables (or predictors),i.e. Xᵢ, while the positive or negative response of the assay is the dependent variable (or response), i.e. Yᵢ .  
loading data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

SEQUENCES_URL = 'https://raw.githubusercontent.com/abidlabs/deep-learning-genomics-primer/master/sequences.txt'

sequences = requests.get(SEQUENCES_URL).text.split('\n')
sequences = list(filter(None, sequences))  # This removes empty sequences.

# Let's print the first few sequences.
pd.DataFrame(sequences, index=np.arange(1, len(sequences)+1),
             columns=['MySequences']).head()
print(pd.DataFrame(sequences, index=np.arange(1, len(sequences)+1),
             columns=['MySequences']).head())
print(len(sequences))

                                         MySequences
1  CCGAGGGCTATGGTTTGGAAGTTAGAACCCTGGGGCTTCTCGCGGA...
2  GAGTTTATATGGCGCGAGCCTAGTGGTTTTTGTACTTGTTTGTCGC...
3  GATCAGTAGGGAAACAAACAGAGGGCCCAGCCACATCTAGCAGGTA...
4  GTCCACGACCGAACTCCCACCTTGACCGCAGAGGTACCACCAGAGC...
5  GGCGACCGAACTCCAACTAGAACCTGCATAACTGGCCTGGGAGATA...
2000


In [11]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
# The LabelEncoder encodes a sequence of bases as a sequence of integers.
integer_encoder = LabelEncoder()
# The OneHotEncoder converts an array of integers to a sparse matrix where
# each row corresponds to one possible value of each feature.
one_hot_encoder = OneHotEncoder(categories=[range(4)])
input_features = []
np.set_printoptions(threshold=40)
for sequence in sequences:

  integer_encoded = integer_encoder.fit_transform(list(sequence))
  #print(integer_encoded)
  integer_encoded = np.array(integer_encoded).reshape(-1, 1)
  one_hot_encoded = one_hot_encoder.fit_transform(integer_encoded)
  if (len(input_features)) % 200 == 0:print('one_hot_encoded:',one_hot_encoded.toarray().T,type(one_hot_encoded))#
  input_features.append(one_hot_encoded.toarray())
  # print('one_hot_encoded.toarray\n',one_hot_encoded.toarray(),type(one_hot_encoded.toarray()))

one_hot_encoded: [[0. 0. 0. ... 1. 0. 0.]
 [1. 1. 0. ... 0. 1. 1.]
 [0. 0. 1. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]] <class 'scipy.sparse._csr.csr_matrix'>
one_hot_encoded: [[0. 0. 1. ... 1. 0. 0.]
 [0. 0. 0. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 0. 0.]
 [1. 1. 0. ... 0. 1. 0.]] <class 'scipy.sparse._csr.csr_matrix'>
one_hot_encoded: [[0. 0. 0. ... 1. 0. 0.]
 [0. 0. 0. ... 0. 1. 1.]
 [1. 0. 1. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]] <class 'scipy.sparse._csr.csr_matrix'>
one_hot_encoded: [[0. 0. 0. ... 0. 0. 0.]
 [1. 0. 0. ... 1. 0. 0.]
 [0. 1. 1. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 1. 0.]] <class 'scipy.sparse._csr.csr_matrix'>
one_hot_encoded: [[0. 1. 1. ... 1. 1. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 1.]] <class 'scipy.sparse._csr.csr_matrix'>
one_hot_encoded: [[0. 0. 1. ... 1. 0. 1.]
 [1. 0. 0. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 1. 0.]] <class 'scipy.sparse._csr.csr_matrix'>
one_hot_encoded: [[0. 0. 0. ... 0. 1. 0.]
 [0. 1. 0. ... 0

Then use above code encode DNA sequecne into : [[0. 1. 0. 0.],[]...,[]],4-dimensional vector as element for each base. 
np.stack() transfer the outermost list into array


In [ ]:
input_features = np.stack(input_features)
print(input_features[1].T)
LABELS_URL = 'https://raw.githubusercontent.com/abidlabs/deep-learning-genomics-primer/master/labels.txt'

labels = requests.get(LABELS_URL).text.split('\n')
labels = list(filter(None, labels))  # removes empty sequences

one_hot_encoder = OneHotEncoder(categories=[range(2)])

labels = np.array(labels).reshape(-1, 1)
# print(labels,type(labels))
print(len(one_hot_encoder.fit_transform(labels).toarray()))
input_labels = one_hot_encoder.fit_transform(labels).toarray()

print('Labels:\n',labels.T)
print('One-hot encoded labels:\n',input_labels.T)

[[0. 1. 0. 0. 0. 0. 1. 0. 1. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 1. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 1. 0. 0. 0. 1. 1. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 1. 0. 0.
  1. 0.]
 [1. 0. 1. 0. 0. 0. 0. 0. 0. 0. 1. 1. 0. 1. 0. 1. 0. 1. 0. 0. 0. 0. 1. 0.
  1. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 1. 0. 1. 0.
  0. 1.]
 [0. 0. 0. 1. 1. 1. 0. 1. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1.
  0. 0. 1. 1. 1. 1. 1. 0. 1. 0. 0. 1. 1. 0. 1. 1. 1. 0. 1. 0. 0. 0. 0. 1.
  0. 0.]]


At each step, a cross-entropy loss is calculated over all the masked k-mers. We optimize DNABERT with AdamW using the following parameters: β_1=0.9,β_2=0.98,ϵ=1e-6  and weight decay as 0.01.  
https://zhuanlan.zhihu.com/p/643452086